# Session 03 — How One-Hot Targets Become a Distribution

Every training row names one observed token. This notebook makes visible how repeated one-hot corrections can nevertheless teach a calibrated non-one-hot next-token distribution.

**Learning cycle:** deep question → prediction → transparent gradient → optimizer trajectory → perturbation → evidence boundary.

## 0. Load the reusable experiment

The reusable computation lives in `src/dongxi_llms/next_token_distribution_lab.py`; this notebook narrates it and exposes its intermediate quantities.

In [ ]:
import math
import sys
from pathlib import Path

import torch

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'src' / 'dongxi_llms').is_dir()
)
sys.path.insert(0, str(repo_root / 'src'))

from dongxi_llms.next_token_distribution_lab import run_experiment

print('Repository:', repo_root)
print('PyTorch:', torch.__version__)

## 1. The apparent paradox

For one fixed context, the dataset contains 70 `dog` targets and 30 `cat` targets. Every individual target is one-hot. Before continuing, explain qualitatively why the model should not converge to 100% `dog`.

<details>
<summary><strong>Reference explanation</strong></summary>

A `dog` example pulls probability toward `dog`, but each `cat` example pulls it back toward `cat`. Assigning zero probability to `cat` would make every observed `cat` infinitely surprising in exact mathematics. Across representative repetitions, the pulls balance at their frequencies—not at the modal token alone.
</details>

## 2. Implement the per-example logit gradient

For probabilities $p$ and one-hot target $q$, implement $\partial L/\partial z=p-q$.

In [ ]:
def logit_gradient(probabilities, one_hot_target):
    # Replace ... after stating what each sign should mean.
    return ...

### Reference solution — one example

In [ ]:
def reference_logit_gradient(probabilities, one_hot_target):
    return probabilities - one_hot_target

p = torch.tensor([0.7, 0.3])
q_dog = torch.tensor([1.0, 0.0])
q_cat = torch.tensor([0.0, 1.0])
g_dog = reference_logit_gradient(p, q_dog)
g_cat = reference_logit_gradient(p, q_cat)
print('dog-example gradient:', g_dog.tolist())
print('cat-example gradient:', g_cat.tolist())
assert torch.allclose(g_dog, torch.tensor([-0.3, 0.3]))
assert torch.allclose(g_cat, torch.tensor([0.7, -0.7]))

**Why the signs matter:** gradient descent moves opposite the gradient. A `dog` example raises the `dog` logit and lowers `cat`; a `cat` example does the reverse. Neither example states the complete distribution by itself.

## 3. Show how competing one-hot gradients balance

At $p=[0.7,0.3]$, combine the two per-example gradients using the observed 70/30 frequencies. Predict whether the result should point toward `dog`, toward `cat`, or nowhere.

In [ ]:
def expected_gradient(dog_frequency, dog_gradient, cat_gradient):
    # Replace ... with the frequency-weighted gradient.
    return ...

### Reference solution — expectation across examples

In [ ]:
def reference_expected_gradient(dog_frequency, dog_gradient, cat_gradient):
    return dog_frequency * dog_gradient + (1 - dog_frequency) * cat_gradient

g_expected = reference_expected_gradient(0.7, g_dog, g_cat)
print('expected gradient:', g_expected.tolist())
assert torch.allclose(g_expected, torch.zeros(2), atol=1e-7)

**Mechanism:** $\mathbb{E}[q]=r$, so $\mathbb{E}[p-q]=p-r$. At $p=r$, individual examples still disagree, but their expected corrections cancel. With small random minibatches, gradients fluctuate around this equilibrium rather than remaining exactly zero.

## 4. Watch two logits learn 70/30

The model has no transformer and no semantic features—only two shared trainable logits. This isolates the optimization mechanism. Predict what should happen to probability, loss, and gradient norm.

In [ ]:
result = run_experiment(steps=500, learning_rate=0.5, seed=42)
print(f"{'step':>5}  {'p(dog)':>10}  {'p(cat)':>10}  {'loss':>10}  {'|grad|':>12}")
for row in result['checkpoints']:
    p_dog, p_cat = row['probabilities']
    print(
        f"{row['step']:5d}  {p_dog:10.7f}  {p_cat:10.7f}  "
        f"{row['mean_cross_entropy']:10.7f}  {row['gradient_norm']:12.3e}"
    )
assert result['all_criteria_passed']

**What converged:** the prediction approaches `[0.7,0.3]`, loss approaches the empirical entropy, and the full-batch gradient approaches zero. The nonzero limiting loss is successful calibrated learning, not residual optimizer failure.

## 5. Recover the learned logit gap

Softmax implies $p_0/p_1=\exp(z_0-z_1)$. Implement the logit difference required by a two-class target distribution.

In [ ]:
def required_logit_gap(probability_0, probability_1):
    return ...

### Reference solution — probability ratio to relative logits

In [ ]:
def reference_required_logit_gap(probability_0, probability_1):
    return math.log(probability_0 / probability_1)

expected_gap = reference_required_logit_gap(0.7, 0.3)
final_logits = result['checkpoints'][-1]['logits']
observed_gap = final_logits[0] - final_logits[1]
print('expected gap:', expected_gap)
print('observed gap:', observed_gap)
assert math.isclose(observed_gap, expected_gap, abs_tol=1e-6)

**Why only the gap is learned:** adding the same constant to both logits changes neither their ratio nor softmax. The experiment's symmetric values near `[0.42365,-0.42365]` are one representative of an infinite family with the same difference.

## 6. Perturb the final distribution

Compare expected cross-entropy for an underconfident prediction, the calibrated prediction, and a mode-seeking prediction. Which should have minimum expected loss?

In [ ]:
target_distribution = torch.tensor([0.7, 0.3])
candidate_predictions = {
    'underconfident': torch.tensor([0.5, 0.5]),
    'calibrated': torch.tensor([0.7, 0.3]),
    'mode-seeking': torch.tensor([0.9, 0.1]),
}
losses = {}
for name, prediction in candidate_predictions.items():
    losses[name] = -(target_distribution * torch.log(prediction)).sum().item()
    print(f'{name:14s}: {losses[name]:.7f}')
assert min(losses, key=losses.get) == 'calibrated'

**Interpretation:** expected cross-entropy is minimized at the complete target distribution, not at the most frequent outcome alone. In $H(r,p)=H(r)+D_{KL}(r\|p)$, changing the model can remove KL mismatch but cannot remove target entropy.

## 7. Evidence boundary

Write one sentence for each: what did this experiment demonstrate, and what would be an unsupported claim about real language models?

<details>
<summary><strong>Reference boundary</strong></summary>

**Supported:** in this controlled two-logit, full-batch system, repeated 70/30 one-hot targets made PyTorch cross-entropy converge to the empirical distribution, and autograd agreed with `p-r`.

**Not supported:** this does not establish that a transformer recovers the true human-language distribution, generalizes to unseen contexts, understands either token, or remains calibrated under dataset and decoding shift.
</details>

## Final synthesis question

At convergence, every individual one-hot example can still have a nonzero gradient and the mean loss can remain positive. Explain why the full-batch expected gradient is nevertheless zero—and why this is successful learning rather than a contradiction.

<details>
<summary><strong>Reference synthesis</strong></summary>

A `dog` sample pulls toward `dog` and a `cat` sample pulls toward `cat`; at `p=r`, their frequency-weighted gradients cancel because $\mathbb{E}[p-q]=p-r=0$. The positive loss equals the uncertainty $H(r)$ that remains even after KL mismatch vanishes. Zero expected gradient says no local change can improve expected cross-entropy under this empirical distribution; it does not say every sampled outcome became certain.
</details>